In [1]:
!pip install requests
!pip install dotenv

In [2]:
import requests
import json
import time
import re  # Regex for extracting Planner Plan ID from URLs
import urllib.parse
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv()

# Access token from .env
ACCESS_TOKEN = os.getenv("ACCESS_TOKEN")

In [3]:
with open("initial_data.json", "r", encoding="utf-8") as file:
    teams_data = json.load(file)


In [4]:
# API Headers
headers = {
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "Content-Type": "application/json"
}


In [ ]:
# Store results
channel_planners_detected = []

def extract_plan_id_from_planner_url(planner_url: str) -> str:
    """
    Extracts the 'real' Plan ID from a Microsoft Teams Planner tab URL.
    1) Decodes the overall URL (which contains a webUrl param).
    2) Finds the path segment after '/PlanViews/' if present.
    3) Otherwise looks for '?planId=' in the decoded string.
    Returns an empty string if nothing is found.
    """
    if not planner_url:
        return ""
    
    # Step 1: Decode the entire Teams URL
    decoded_url = urllib.parse.unquote(planner_url)
    
    # The real plan URL is inside the webUrl=... part. Let's safely extract it:
    # Example: ...?webUrl=<ENCODED_TASKS_URL>&label=...
    parsed = urllib.parse.urlparse(decoded_url)
    qs = urllib.parse.parse_qs(parsed.query)
    
    # webUrl might be in qs["webUrl"] if it exists
    # It's typically a list with one element if present
    if "webUrl" in qs and qs["webUrl"]:
        real_planner_link = qs["webUrl"][0]
    else:
        # If for some reason webUrl is missing, just fallback to the full decoded_url
        real_planner_link = decoded_url

    # Step 2: In that real Planner link, look for '/PlanViews/XXXXX?'
    plan_id_match = re.search(r'/PlanViews/([^?&]+)', real_planner_link)
    if plan_id_match:
        return plan_id_match.group(1)

    # Step 3 (fallback): Look for planId=XXXX if the above fails
    plan_id_match = re.search(r'[?&]planId=([^&]+)', real_planner_link)
    if plan_id_match:
        return plan_id_match.group(1)

    return ""  # If all else fails, return empty string


# Function to clean "tt." formatted Planner IDs (removing incorrect prefixes)
def clean_plan_id(plan_id):
    """Extracts the correct ID from messy tt. planner_id values"""
    if not plan_id:
        return None

    match = re.search(r"tt\.([a-zA-Z0-9-_]+)", plan_id)  # Extract after "tt."
    return f"tt.{match.group(1)}" if match else None  # Return cleaned Planner ID

# Function to check if a tab is a valid Planner Tab
def is_planner_tab(tab_name, planner_id, planner_url):
    """Determines if a tab is a valid Planner tab"""
    if planner_id and planner_id.startswith("tt."):  # Confirm it's a valid Planner ID
        return True
    
    if "planner" in tab_name.lower() or "tasks" in tab_name.lower():
        return True  # Likely a Planner tab

    if planner_url and "tasks.office.com" in planner_url:
        return True  # The URL confirms it's Planner

    return False  # Not a Planner tab

# Loop through each team
for team in teams_data["teams"]:
    team_id = team["id"]
    project_name = team["project_name"]

    # print(f"\n🔍 Fetching channels for team: {project_name} ({team_id})")

    # Step 1: Get all channels in the team
    channels_url = f"https://graph.microsoft.com/v1.0/teams/{team_id}/channels"
    channels_response = requests.get(channels_url, headers=headers)

    if channels_response.status_code == 200:
        channels = channels_response.json().get("value", [])
    else:
        continue  # Skip this team if there's an error

    # Step 2: Check each channel for a Planner tab
    for channel in channels:
        channel_name = channel["displayName"]
        channel_id = channel["id"]
        detected_planner_tabs = []

        # Get tabs in the channel
        tabs_url = f"https://graph.microsoft.com/v1.0/teams/{team_id}/channels/{channel_id}/tabs"
        tabs_response = requests.get(tabs_url, headers=headers)

        if tabs_response.status_code == 200:
            tabs = tabs_response.json().get("value", [])

            # Loop through each tab
            for tab in tabs:
                tab_name = tab.get("displayName", "")
                planner_id = tab.get("configuration", {}).get("entityId")
                planner_url = tab.get("webUrl")

                # Extract the real Plan ID from the URL
                extracted_plan_id = extract_plan_id_from_planner_url(planner_url)

                # If a valid Planner ID exists in the URL, use that instead
                if extracted_plan_id:
                    planner_id = extracted_plan_id
                else:
                    planner_id = clean_plan_id(planner_id)  # Clean existing planner_id

                # Check if this is a valid Planner tab
                if planner_id and is_planner_tab(tab_name, planner_id, planner_url):
                    detected_planner_tabs.append({
                        "tab_name": tab_name,
                        "planner_id": planner_id,  # Now correctly extracted from URL
                        "planner_url": planner_url,
                    })

                    # Print only when a valid Planner tab is found
                    # print(f"✅ Planner Tab Found: {tab_name} ({planner_id})")

        # Store detected Planner Tabs only if any were found
        if detected_planner_tabs:
            channel_planners_detected.append({
                "project_name": project_name,
                "team_id": team_id,
                "channel_name": channel_name,
                "channel_id": channel_id,
                "planner_tabs": detected_planner_tabs  # Storing only valid "tt." prefixed IDs
            })

    time.sleep(1)  # Avoid hitting API rate limits

In [46]:
# Save results to JSON file
with open("teams_data.json", "w") as file:
    json.dump(channel_planners_detected, file, indent=4)

print("\n🎯 Planner Tabs Detection Saved to teams_data.json")


🎯 Planner Tabs Detection Saved to teams_data.json
